# 2장 복습 — 문제

2장 퍼셉트론을 다른 각도에서 다시 풀어 보는 복습 문제.
- Part A 매개변수 · **Part B 에러 찾기** · Part C 한계와 경계선 · Part D 다층 퍼셉트론
- 뒤 문제가 앞 문제의 함수를 쓰므로 **위에서부터 순서대로** 실행
- 막히면 Obsidian `2장 퍼셉트론` 노트 → 그래도 막히면 `ch02_review_solutions.ipynb`
- 커널은 `.venv`, **0. 준비 셀부터** 실행

## 0. 준비
먼저 실행하자. NumPy, matplotlib, 진리표 채점 도우미를 불러온다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 진리표 채점 도우미
def check_gate(gate, expected, name):
    inputs = [(0, 0), (1, 0), (0, 1), (1, 1)]
    ok = True
    for (x1, x2), want in zip(inputs, expected):
        got = gate(x1, x2)
        mark = '✅' if got == want else '❌'
        ok = ok and got == want
        print(f'{name}({x1}, {x2}) = {got}   (정답 {want}) {mark}')
    assert ok, f'{name} 진리표가 맞지 않아요'
    print('통과!')


# Part A. 퍼셉트론과 매개변수
## A-1. 편향 버전으로 손 계산
$x = (1, 1)$, $w = (0.6, -0.4)$, $b = -0.1$일 때
- `tmp`: $b + w_1x_1 + w_2x_2$
- `out`: 출력

**종이에 먼저 계산**한 뒤, 이번에는 결과가 아니라 **계산식을 코드로** 쓰자. (NumPy 사용)

In [ ]:
x = np.array([1, 1])
w = np.array([0.6, -0.4])
b = -0.1
tmp = None  # 여기에 작성 (식으로!)
out = None


In [ ]:
# 확인
assert np.isclose(tmp, 0.1) and out == 1
print('통과!')

## A-2. 가중치가 다를 때 AND 만들기
가중치가 $w = (0.3, 0.6)$으로 **고정**되어 있다. 편향 `b`만 골라서 AND를 만들자.
> 힌트: 입력 4가지의 $w_1x_1 + w_2x_2$ 값을 먼저 적어 보자. AND가 되려면 θ가 어느 범위여야 할까? 그리고 $b = -\theta$.

In [ ]:
b_and = None  # 여기에 작성

def AND(x1, x2):
    x = np.array([x1, x2])
    w = np.array([0.3, 0.6])
    return 1 if np.sum(w * x) + b_and > 0 else 0


In [ ]:
# 확인
check_gate(AND, [0, 0, 0, 1], 'AND')

## A-3. 범용 퍼셉트론 함수
**구조는 같고 매개변수만 다르다**는 것을 코드로 표현해 보자.
1. `perceptron(x1, x2, w, b)`: 가중치 배열 `w`와 편향 `b`를 받아 0 또는 1을 돌려주는 함수
2. 이 함수 **하나만** 써서 AND2, NAND2, OR2를 한 줄씩 만들기 (매개변수는 자유)

In [ ]:
def perceptron(x1, x2, w, b):
    pass  # 여기에 작성

def AND2(x1, x2):  return None  # perceptron(...)을 불러서
def NAND2(x1, x2): return None
def OR2(x1, x2):   return None


In [ ]:
# 확인
check_gate(AND2, [0, 0, 0, 1], 'AND2')
check_gate(NAND2, [1, 1, 1, 0], 'NAND2')
check_gate(OR2, [0, 1, 1, 1], 'OR2')

## A-4. 새 게이트: NOR
NOR = OR의 반대 (둘 다 0일 때만 1)

| $x_1$ | $x_2$ | NOR |
|:-:|:-:|:-:|
| 0 | 0 | 1 |
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 1 | 0 |

A-3의 `perceptron`으로 만들자.
> 힌트: NAND를 AND에서 만든 방법을 떠올리기.

In [ ]:
def NOR(x1, x2):
    pass  # 여기에 작성


In [ ]:
# 확인
check_gate(NOR, [1, 0, 0, 0], 'NOR')

## A-5. 입력이 하나인 NOT
NOT: 0 → 1, 1 → 0. 입력이 $x$ 하나뿐인 퍼셉트론 $y = [wx + b > 0]$으로 만들자.
> 힌트: $x=0$이면 $b$만 남는다. $x=1$이면 $w+b$.

In [ ]:
def NOT(x):
    w = None  # 여기에 작성
    b = None
    return 1 if w * x + b > 0 else 0


In [ ]:
# 확인
assert NOT(0) == 1 and NOT(1) == 0
print('통과!')

# Part B. 에러 찾기 🔍
2장 연습 때 실제로 했던 실수가 숨어 있다. 실행해서 에러 메시지를 읽고 고치자.

## B-1. NAND (실수 3개)

In [ ]:
def NAND(x1, x2):
    x = np.aaray([x1, x2])
    w = np.array([-0.5, -0.5])
    b = 0.7

        return 1 if np.sum(w * x) + b > 0


In [ ]:
# 확인
check_gate(NAND, [1, 1, 1, 0], 'NAND')

## B-2. 진리표 딕셔너리 다루기 (실수 1개)
`and_out`의 각 점에 대해, 출력이 1이면 `'^'`, 0이면 `'o'`를 `markers`에 순서대로 담자.

In [ ]:
and_out = {(0, 0): 0, (1, 0): 0, (0, 1): 0, (1, 1): 1}
markers = []
for (x1, x2) in and_out:
    markers.append('^' if y == 1 else 'o')


In [ ]:
# 확인
assert markers == ['o', 'o', 'o', '^']
print('통과!')

## B-3. XOR (설계 실수 1개)
아래 XOR은 **결과는 맞는다.** 그런데 2.5절 '다층 퍼셉트론'이라고 할 수 없는 부분이 있다. 어디일까?
마지막 단계를 **퍼셉트론 게이트로** 바꿔서 고치자. (A-3의 AND2, NAND2, OR2 사용)

In [ ]:
def XOR(x1, x2):
    s1 = NAND2(x1, x2)
    s2 = OR2(x1, x2)
    return 1 if (s1 == 1 and s2 == 1) else 0


In [ ]:
# 확인
import inspect, ast
check_gate(XOR, [0, 1, 1, 0], 'XOR')
tree = ast.parse(inspect.getsource(XOR))
assert not any(isinstance(n, ast.BoolOp) for n in ast.walk(tree)), '마지막 단계에 파이썬 and/or 연산자가 남아 있어요'
assert any(isinstance(n, ast.Name) and n.id == 'AND2' for n in ast.walk(tree)), 'AND2 게이트를 써 주세요'
print('통과!')

# Part C. 퍼셉트론의 한계
## C-1. 경계선의 식 구하기
경계는 $b + w_1x_1 + w_2x_2 = 0$ 이다. 이걸 $x_2$에 대해 정리하면
$$x_2 = -\frac{w_1}{w_2}x_1 - \frac{b}{w_2}$$
`boundary(w, b)` 함수가 이 직선의 `(기울기, y절편)`을 돌려주도록 만들자.

In [ ]:
def boundary(w, b):
    slope = None      # 여기에 작성
    intercept = None
    return slope, intercept


In [ ]:
# 확인
assert np.allclose(boundary(np.array([0.5, 0.5]), -0.2), (-1, 0.4))     # OR
assert np.allclose(boundary(np.array([-0.5, -0.5]), 0.7), (-1, 1.4))   # NAND
assert np.allclose(boundary(np.array([0.3, 0.6]), -0.75), (-0.5, 1.25))  # A-2의 AND
print('통과!')

## C-2. 경계선 그리기
A-2의 AND(`w = [0.3, 0.6]`, `b = b_and`)에 대해
1. 4개 점을 출력에 따라 `'o'` / `'^'`로 찍고
2. C-1의 `boundary`로 구한 경계선을 점선으로 그리자.

세모 하나만 직선 반대편에 있으면 성공.

In [ ]:
# 여기에 작성

plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.show()


## C-3. 직선 하나로 나눌 수 있을까?
각 게이트가 **단층 퍼셉트론(직선 하나)**으로 만들 수 있으면 `True`, 아니면 `False`를 적자.
XNOR은 XOR의 반대(두 입력이 같으면 1)다.
> 헷갈리면 점 4개를 종이에 찍어 보자.

In [ ]:
separable = {
    'AND':  None,
    'NAND': None,
    'OR':   None,
    'NOR':  None,
    'XOR':  None,
    'XNOR': None,
}


In [ ]:
# 확인
assert separable == {'AND': True, 'NAND': True, 'OR': True, 'NOR': True, 'XOR': False, 'XNOR': False}
print('통과!')

# Part D. 다층 퍼셉트론
## D-1. 다른 조합으로 XOR 만들기
XOR은 "$x_1$만 1" **또는** "$x_2$만 1"인 경우다.
$$\text{XOR} = \text{OR}(\ \text{AND}(x_1, \text{NOT}\ x_2),\ \text{AND}(\text{NOT}\ x_1, x_2)\ )$$
A-3의 AND2, OR2와 A-5의 NOT으로 `XOR_v2`를 만들자.

In [ ]:
def XOR_v2(x1, x2):
    pass  # 여기에 작성


In [ ]:
# 확인
check_gate(XOR_v2, [0, 1, 1, 0], 'XOR_v2')

## D-2. XNOR 만들기
XNOR = 두 입력이 **같으면** 1. 이미 만든 게이트를 조합해서 만들자. (방법은 여러 가지)

In [ ]:
def XNOR(x1, x2):
    pass  # 여기에 작성


In [ ]:
# 확인
check_gate(XNOR, [1, 0, 0, 1], 'XNOR')

## D-3. 층 세기
이 책의 기준(**가중치가 있는 층의 수**)으로 세어 보자.
- `layers_xor`: NAND·OR → AND 구조의 XOR
- `layers_xnor`: D-2 방법 1 (XOR 뒤에 NOT)

In [ ]:
layers_xor = None
layers_xnor = None


In [ ]:
# 확인
assert layers_xor == 2 and layers_xnor == 3
print('통과!')

## 생각해 보기
1. A-3에서 `perceptron` 함수 하나로 AND, NAND, OR, NOR를 모두 만들었다. 이것이 4장 '학습'과 어떻게 이어질까?
2. C-3에서 False였던 게이트도 Part D에서는 만들 수 있었다. 무엇이 달라졌을까?

In [ ]:
# 1. 내 생각:

# 2. 내 생각:
